# ARC-JEPA: inference (neural-guided program search -> submission.json)

Loads the offline package `arc_jepa_pkg` exported by `poby7722/arc-jepa-train` (or runs the pure symbolic
search when it is missing), solves every task with `arcjepa.search.solver.solve_task` and writes
`submission.json` (two attempts per test input). Fallback attempts are written before any search and the file is
rewritten every 60 s, so a valid submission always exists.

* Competition rerun (`KAGGLE_IS_COMPETITION_RERUN`): `arc-agi_test_challenges.json`, global budget
  `ARCJEPA_GLOBAL_HOURS` (default 11 h), fair-share seconds per task, shortest task first.
* Interactive / commit run: dev mode on the 120 public evaluation tasks (`ARCJEPA_DEV_HOURS`, default 1 h,
  `ARCJEPA_DEV_MAX_TASKS` to cap), scored with the competition metric. The last cell is a **gate**: it raises
  (the commit fails and cannot be submitted) when the code dataset is missing, the trained package is not found
  or is a debug/smoke package, or no solver process could load it. A competition rerun never fails on these.

Other switches: `ARCJEPA_WORKERS` (default `auto`), `ARCJEPA_THREADS` (torch threads per solver process, default
auto = CPUs / workers), `ARCJEPA_PKG` (explicit package folder),
`ARCJEPA_MAX_TASK_SECONDS` (default `auto` = max(1800, 2 x budget x workers / tasks)); local testing only:
`ARCJEPA_ALLOW_SYMBOLIC=1`, `ARCJEPA_ALLOW_DEBUG_PKG=1` (the gate reports instead of raising).

In [ ]:
import glob
import json
import os
import shutil
import sys
import time
import traceback

T0 = time.time()
IS_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
GLOBAL_BUDGET_S = float(os.environ.get("ARCJEPA_GLOBAL_HOURS", "11")) * 3600.0
DEV_HOURS = float(os.environ.get("ARCJEPA_DEV_HOURS", "1.0"))
DEV_MAX_TASKS = int(os.environ.get("ARCJEPA_DEV_MAX_TASKS", "0") or 0)
MAX_TASK_ENV = os.environ.get("ARCJEPA_MAX_TASK_SECONDS", "auto")
ALLOW_SYMBOLIC = os.environ.get("ARCJEPA_ALLOW_SYMBOLIC", "0") == "1"    # local tests only (Kaggle sets no env)
ALLOW_DEBUG_PKG = os.environ.get("ARCJEPA_ALLOW_DEBUG_PKG", "0") == "1"  # local tests only
WORKERS_ENV = os.environ.get("ARCJEPA_WORKERS", "auto")
THREADS = int(os.environ.get("ARCJEPA_THREADS", "0") or 0) or None  # torch threads per solver process (auto)
WORK = os.environ.get("ARCJEPA_WORK", "/kaggle/working")
SUBMISSION = os.path.join(WORK, "submission.json")
COMP_DIRS = [d for d in (os.environ.get("ARCJEPA_COMP_DIR", ""),
                         "/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
                         "/kaggle/input/arc-prize-2026-arc-agi-2") if d]
RESERVE_S = min(600.0, 0.05 * GLOBAL_BUDGET_S)  # kept for the final write + validation
os.makedirs(WORK, exist_ok=True)
print("mode:", "COMPETITION RERUN" if IS_RERUN else "dev (public evaluation)")

In [ ]:
# ---- submission validation + fallbacks (inlined from kaggle/validate_submission.py)
import json
import os
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence

ATTEMPT_KEYS = ("attempt_1", "attempt_2")
MAX_GRID_SIDE = 30


def grid_error(g: Any) -> Optional[str]:
    """``None`` when ``g`` is a valid ARC grid, else a short reason."""
    if not isinstance(g, list) or not g:
        return "grid is not a non-empty list"
    if len(g) > MAX_GRID_SIDE:
        return "grid has %d rows (> %d)" % (len(g), MAX_GRID_SIDE)
    if not isinstance(g[0], list) or not g[0]:
        return "first row is not a non-empty list"
    w = len(g[0])
    if w > MAX_GRID_SIDE:
        return "grid has %d columns (> %d)" % (w, MAX_GRID_SIDE)
    for r, row in enumerate(g):
        if not isinstance(row, list) or len(row) != w:
            return "row %d is not a list of length %d" % (r, w)
        for v in row:
            if isinstance(v, bool) or not isinstance(v, int) or v < 0 or v > 9:
                return "row %d has a value that is not an int 0-9: %r" % (r, v)
    return None


def validate_submission(sub: Any, challenges: Optional[Dict[str, Any]] = None,
                        task_ids: Optional[Sequence[str]] = None) -> List[str]:
    """All format errors of ``sub`` (empty list = valid).

    With ``challenges`` (the challenges JSON dict) every task id must be present, no extra ids are allowed and
    each task must have exactly one attempts dict per test input. ``task_ids`` alone only checks presence.
    """
    errs: List[str] = []
    if not isinstance(sub, dict):
        return ["submission is not a JSON object"]
    if not sub:
        errs.append("submission is empty")
    expected: Dict[str, Optional[int]] = {}
    if challenges is not None:
        for tid, t in challenges.items():
            n = len(t.get("test", [])) if isinstance(t, dict) else None
            expected[str(tid)] = n
    for tid in task_ids or ():
        expected.setdefault(str(tid), None)
    for tid in expected:
        if tid not in sub:
            errs.append("task %s missing" % tid)
    if challenges is not None:
        for tid in sub:
            if tid not in expected:
                errs.append("task %s is not in the challenges" % tid)
    for tid, entries in sub.items():
        if not isinstance(tid, str):
            errs.append("task id %r is not a string" % (tid,))
        if not isinstance(entries, list) or not entries:
            errs.append("task %s: value is not a non-empty list" % tid)
            continue
        n = expected.get(str(tid))
        if n is not None and len(entries) != n:
            errs.append("task %s: %d attempt dicts for %d test inputs" % (tid, len(entries), n))
        for i, e in enumerate(entries):
            if not isinstance(e, dict):
                errs.append("task %s test %d: entry is not an object" % (tid, i))
                continue
            keys = set(e)
            missing = [k for k in ATTEMPT_KEYS if k not in keys]
            extra = sorted(str(k) for k in keys - set(ATTEMPT_KEYS))
            if missing:
                errs.append("task %s test %d: missing %s" % (tid, i, ",".join(missing)))
            if extra:
                errs.append("task %s test %d: unexpected keys %s" % (tid, i, ",".join(extra)))
            for k in ATTEMPT_KEYS:
                if k in e:
                    ge = grid_error(e[k])
                    if ge:
                        errs.append("task %s test %d %s: %s" % (tid, i, k, ge))
    return errs


def validate_file(path: str, challenges_path: Optional[str] = None) -> List[str]:
    """Errors of the submission file at ``path`` (JSON parse errors included)."""
    try:
        with open(path, "r", encoding="utf-8") as fh:
            sub = json.load(fh)
    except (OSError, ValueError) as exc:
        return ["cannot read %s: %s" % (path, exc)]
    challenges = None
    if challenges_path:
        with open(challenges_path, "r", encoding="utf-8") as fh:
            challenges = json.load(fh)
    return validate_submission(sub, challenges)


def predicted_output_shape(demo: Sequence[Any], test_input: Any) -> Optional[Any]:
    """Output shape implied by valid demo (input, output) grids: same as input, a consistent integer scaling or a
    constant shape; else None (same rule as arcjepa.search.diversity.predict_output_shape)."""
    if not demo or grid_error(test_input) is not None:
        return None
    ih, iw = len(test_input), len(test_input[0])
    shapes = [((len(i), len(i[0])), (len(o), len(o[0]))) for i, o in demo]
    if all(a == b for a, b in shapes):
        return (ih, iw)
    fr = {(o[0] / i[0], o[1] / i[1]) for i, o in shapes}
    if len(fr) == 1:
        a, b = next(iter(fr))
        h, w = ih * a, iw * b
        if abs(h - round(h)) < 1e-9 and abs(w - round(w)) < 1e-9 and 1 <= round(h) <= 30 and 1 <= round(w) <= 30:
            return (int(round(h)), int(round(w)))
    outs = {o for _, o in shapes}
    return next(iter(outs)) if len(outs) == 1 else None


def fallback_attempts(task: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Always-valid attempts for every test input of a challenges task dict, without any search: the first two
    distinct grids of [constant demo output (all demo outputs equal); the identity for identity tasks (every demo
    output equals its input); the predicted output shape (else the most common demo output shape) filled with the
    most common and the second most common demo output colour; the identity; [[0]]]. The identity is a last
    resort: no test output of the 1,000 public training tasks equals its input. Same rule as
    arcjepa.utils.kaggle_submit_runner.fallback_attempts."""
    demo = [(p["input"], p["output"]) for p in task.get("train", []) or []
            if isinstance(p, dict) and grid_error(p.get("input")) is None and grid_error(p.get("output")) is None]
    outs = [o for _, o in demo]
    ident_ok = bool(demo) and all(i == o for i, o in demo)
    colours = [c for c, _ in Counter(v for o in outs for row in o for v in row).most_common(2)]
    common = Counter((len(o), len(o[0])) for o in outs).most_common(1)[0][0] if outs else None
    out: List[Dict[str, Any]] = []
    for tp in task.get("test", []) or [{}]:
        g = tp.get("input") if isinstance(tp, dict) else None
        valid_in = grid_error(g) is None
        opts: List[Any] = []
        if outs and all(o == outs[0] for o in outs):
            opts.append(outs[0])
        if valid_in and ident_ok:
            opts.append(g)
        if outs:
            shape = (predicted_output_shape(demo, g) if valid_in else None) or common
            opts.extend([[int(c)] * shape[1] for _ in range(shape[0])] for c in colours)
        if valid_in:
            opts.append(g)
        opts.append([[0]])
        a1 = opts[0]
        a2 = next((o for o in opts[1:] if o != a1), a1)
        out.append({"attempt_1": [list(r) for r in a1], "attempt_2": [list(r) for r in a2]})
    return out


def fallback_submission(challenges: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """Fallback attempts for every task of ``challenges``."""
    return {str(tid): fallback_attempts(t if isinstance(t, dict) else {}) for tid, t in challenges.items()}


def repair_submission(sub: Any, challenges: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """A valid submission for ``challenges``: every valid attempt of ``sub`` is kept, everything missing or
    malformed is replaced by the fallback attempt."""
    fb = fallback_submission(challenges)
    sub = sub if isinstance(sub, dict) else {}
    out: Dict[str, List[Dict[str, Any]]] = {}
    for tid, fallback in fb.items():
        got = sub.get(tid)
        entries: List[Dict[str, Any]] = []
        for i, f in enumerate(fallback):
            e = got[i] if isinstance(got, list) and i < len(got) and isinstance(got[i], dict) else {}
            entries.append({k: (e[k] if k in e and grid_error(e[k]) is None else f[k]) for k in ATTEMPT_KEYS})
        out[tid] = entries
    return out


def score_submission(sub: Dict[str, Any], solutions: Dict[str, List[Any]]) -> float:
    """Competition metric (stdlib copy of ``arcjepa.eval.evaluate.competition_score``): mean over the tasks of
    ``solutions`` of the fraction of test outputs matched by attempt_1 or attempt_2; missing tasks score 0."""
    if not solutions:
        return 0.0
    total = 0.0
    for tid, truths in solutions.items():
        got = sub.get(tid) if isinstance(sub, dict) else None
        got = got if isinstance(got, list) else []
        hits = 0
        for i, t in enumerate(truths):
            e = got[i] if i < len(got) and isinstance(got[i], dict) else {}
            hits += int(bool(t) and any(e.get(k) == t for k in ATTEMPT_KEYS))
        total += hits / float(len(truths)) if truths else 0.0
    return total / len(solutions)


def write_json_atomic(obj: Any, path: str) -> None:
    """Write ``obj`` as JSON to ``path`` via a temporary file + rename (never leaves a half-written file)."""
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, separators=(",", ":"))
    os.replace(tmp, path)

In [ ]:
# ---- challenge files; fallback submission.json is written FIRST


def find_comp_file(name):
    for d in COMP_DIRS:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    hits = sorted(glob.glob(os.path.join("/kaggle/input", "*", name)) + glob.glob(os.path.join("/kaggle/input", "*", "*", name)))
    return hits[0] if hits else None


def load_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


TEST_PATH = find_comp_file("arc-agi_test_challenges.json")
test_challenges = load_json(TEST_PATH) if TEST_PATH else {}
if test_challenges:
    write_json_atomic(fallback_submission(test_challenges), SUBMISSION)
    print("fallback submission written for %d test tasks (%d test inputs)" % (
        len(test_challenges), sum(len(t.get("test", [])) for t in test_challenges.values())))
else:
    print("WARNING: arc-agi_test_challenges.json not found under", COMP_DIRS)

In [ ]:
# ---- locate the ARC-JEPA code dataset, copy it to /kaggle/working/arc-jepa, put it on sys.path
CODE_CANDIDATES = [os.environ.get("ARCJEPA_CODE", ""), "/kaggle/input/datasets/poby7722/arc-jepa-code",
                   "/kaggle/input/arc-jepa-code"]
CODE_DST = os.path.join(WORK, "arc-jepa")


def find_code_root(cands):
    """First folder holding arcjepa/__init__.py: the candidates (up to two levels deep), then /kaggle/input."""
    for base in [c for c in cands if c and os.path.isdir(c)]:
        for pat in ("", "*", "*/*"):
            for d in (sorted(glob.glob(os.path.join(base, pat))) if pat else [base]):
                if os.path.isfile(os.path.join(d, "arcjepa", "__init__.py")):
                    return d
    hits = sorted(glob.glob("/kaggle/input/**/arcjepa/__init__.py", recursive=True))
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None


CODE_SRC = None
CODE_PATH = None
CODE_OK = False
try:
    CODE_SRC = find_code_root(CODE_CANDIDATES)
    if CODE_SRC:
        CODE_PATH = CODE_DST
        if os.path.abspath(CODE_SRC) != os.path.abspath(CODE_DST):
            try:
                shutil.copytree(CODE_SRC, CODE_DST, dirs_exist_ok=True,
                                ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".pytest_cache", "runs", ".git"))
            except Exception as exc:  # importing from the read-only mount works too
                print("copying the code failed (%r); importing it from %s" % (exc, CODE_SRC))
                CODE_PATH = CODE_SRC
        if CODE_PATH not in sys.path:
            sys.path.insert(0, CODE_PATH)
        os.environ["PYTHONPATH"] = CODE_PATH + os.pathsep + os.environ.get("PYTHONPATH", "")  # spawn workers
        import arcjepa  # noqa: F401

        CODE_OK = True
except Exception:
    traceback.print_exc()
print("code source:", CODE_SRC, "-> imported from", CODE_PATH, "| import ok:", CODE_OK)

# ---- the episodes dataset (task mirror; dev mode reads tasks/eval_public.jsonl when the competition files lack it)
EPISODES_ROOT = None
for base in [os.environ.get("ARCJEPA_DATA", ""), "/kaggle/input/datasets/poby7722/arc-agi-2-jepa-episodes",
             "/kaggle/input/arc-agi-2-jepa-episodes"]:
    if not base or not os.path.isdir(base):
        continue
    hits = sorted(glob.glob(os.path.join(base, "tasks", "train.jsonl")) + glob.glob(os.path.join(base, "*", "tasks", "train.jsonl")))
    if hits:
        EPISODES_ROOT = os.path.dirname(os.path.dirname(hits[0]))
        break
if EPISODES_ROOT:
    os.environ["ARCJEPA_DATA"] = EPISODES_ROOT
print("episodes dataset:", EPISODES_ROOT)

In [ ]:
# ---- model package (kernel output of poby7722/arc-jepa-train, mounted under /kaggle/input); symbolic mode when
# missing. Explicit mounts first, then a walk for arc_jepa_pkg folders (config.json + model.safetensors).
PKG_DIR = None
PKG_INFO = {}
RUNNER_OK = False
if CODE_OK:
    try:
        from arcjepa.utils.kaggle_submit_runner import (RunnerConfig, find_package, load_model_package,
                                                        package_info, run_submission)

        RUNNER_OK = True
        PKG_DIR = find_package([os.environ.get("ARCJEPA_PKG", ""),
                                "/kaggle/input/notebooks/poby7722/arc-jepa-train/arc_jepa_pkg",
                                "/kaggle/input/arc-jepa-train/arc_jepa_pkg",
                                "/kaggle/input/kernels/poby7722/arc-jepa-train/arc_jepa_pkg"],
                               search_roots=["/kaggle/input"])
        PKG_INFO = package_info(PKG_DIR) if PKG_DIR else {}
    except Exception:
        traceback.print_exc()
try:
    import torch

    N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    N_GPU = 0
N_CPU = os.cpu_count() or 1
if WORKERS_ENV == "auto":
    N_WORKERS = max(1, min(16, N_CPU - 1))
else:
    N_WORKERS = max(0, int(WORKERS_ENV))
if N_WORKERS > 0 and "__file__" in globals():
    # executed as a plain script (local simulation only): spawn workers would re-run this unguarded script.
    # Kaggle runs the notebook in a Jupyter kernel (no __file__), where the worker pool is used.
    print("script mode: solving in-process instead of %d spawn workers" % N_WORKERS)
    N_WORKERS = 0
DEVICES = ["cuda:%d" % (i % N_GPU) for i in range(max(1, N_WORKERS))] if (N_GPU and PKG_DIR) else None


def task_cap_seconds(budget_s, n_tasks):
    """Per-task budget cap: ARCJEPA_MAX_TASK_SECONDS, or auto = max(1800, 2 x budget x workers / tasks) so the
    cap never leaves the global budget idle when there are many workers."""
    if MAX_TASK_ENV != "auto":
        return float(MAX_TASK_ENV)
    return max(1800.0, 2.0 * budget_s * max(1, N_WORKERS) / max(1, n_tasks))


print("code:", CODE_OK, "| runner:", RUNNER_OK, "| package:", PKG_DIR or "none (symbolic mode)",
      "| GPUs", N_GPU, "| CPUs", N_CPU, "| workers", N_WORKERS)
if PKG_DIR:
    _c = PKG_INFO.get("created_unix")
    print("package created", time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime(_c)) if _c else "?",
          "| parameters", PKG_INFO.get("n_parameters"), "| training config", PKG_INFO.get("config_path"))
    if PKG_INFO.get("debug"):
        print("WARNING: this is a DEBUG (smoke) package: the training kernel did not run the full config")

In [ ]:
# ---- competition rerun: solve the hidden test tasks within the global budget
RERUN_SUMMARY = None
if IS_RERUN and RUNNER_OK and test_challenges:
    try:
        cfg = RunnerConfig(total_seconds=GLOBAL_BUDGET_S, reserve_seconds=RESERVE_S, min_task_seconds=2.0,
                           max_task_seconds=task_cap_seconds(GLOBAL_BUDGET_S, len(test_challenges)),
                           rewrite_every_s=60.0, workers=N_WORKERS, devices=DEVICES, threads_per_worker=THREADS)
        RERUN_SUMMARY = run_submission(test_challenges, SUBMISSION, pkg_dir=PKG_DIR, cfg=cfg,
                                       initial=load_json(SUBMISSION), start_time=T0,
                                       diagnostics_path=os.path.join(WORK, "rerun_diagnostics.json"))
        print({k: v for k, v in RERUN_SUMMARY.items() if k not in ("diagnostics", "unstarted")})
        if PKG_DIR and not RERUN_SUMMARY.get("model_loaded"):
            print("WARNING: package %s was found but no worker loaded it: symbolic search only" % PKG_DIR)
    except Exception:
        traceback.print_exc()
elif IS_RERUN:
    print("rerun without a working solver: the fallback submission stands")

In [ ]:
# ---- dev mode: the 120 public evaluation tasks, scored with the competition metric
DEV_RESULT = None
DEV_SUMMARY = None
if not IS_RERUN:
    eval_ch, eval_sol = {}, {}
    ch_path = find_comp_file("arc-agi_evaluation_challenges.json")
    sol_path = find_comp_file("arc-agi_evaluation_solutions.json")
    if ch_path and sol_path:
        eval_ch, eval_sol = load_json(ch_path), load_json(sol_path)
    elif EPISODES_ROOT and os.path.isfile(os.path.join(EPISODES_ROOT, "tasks", "eval_public.jsonl")):
        with open(os.path.join(EPISODES_ROOT, "tasks", "eval_public.jsonl"), encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    row = json.loads(line)
                    eval_ch[row["task_id"]] = {"train": row["train"], "test": [{"input": p["input"]} for p in row["test"]]}
                    eval_sol[row["task_id"]] = [p["output"] for p in row["test"]]
    ids = sorted(eval_ch)[:DEV_MAX_TASKS] if DEV_MAX_TASKS else sorted(eval_ch)
    eval_ch = {t: eval_ch[t] for t in ids}
    eval_sol = {t: eval_sol[t] for t in ids if t in eval_sol}
    print("dev tasks:", len(eval_ch))
    dev_sub_path = os.path.join(WORK, "dev_eval_submission.json")
    if eval_ch and RUNNER_OK:
        try:
            cfg = RunnerConfig(total_seconds=DEV_HOURS * 3600.0, reserve_seconds=min(30.0, 0.05 * DEV_HOURS * 3600.0),
                               min_task_seconds=2.0,
                               max_task_seconds=task_cap_seconds(DEV_HOURS * 3600.0, len(eval_ch)),
                               rewrite_every_s=60.0, workers=N_WORKERS, devices=DEVICES, threads_per_worker=THREADS)
            dev_summary = run_submission(eval_ch, dev_sub_path, pkg_dir=PKG_DIR, cfg=cfg, start_time=time.time(),
                                         diagnostics_path=os.path.join(WORK, "dev_run_diagnostics.json"))
            DEV_SUMMARY = {k: v for k, v in dev_summary.items() if k != "diagnostics"}
            print("dev run: model loaded by %s of the workers | pool restarts %d | quarantined %s" % (
                dev_summary.get("model_loaded_fraction"), dev_summary.get("pool_restarts", 0),
                dev_summary.get("quarantined")))
            dev_sub = load_json(dev_sub_path)
            from arcjepa.core.types import task_from_json
            from arcjepa.eval.evaluate import competition_score, evaluate, replay_solver

            tasks = {}
            for t, d in eval_ch.items():
                tests = [{"input": tp["input"], "output": o} for tp, o in zip(d["test"], eval_sol.get(t, []))]
                tasks[t] = task_from_json(t, {"train": d["train"], "test": tests})
            DEV_RESULT = evaluate(tasks, replay_solver(dev_sub, dev_summary["diagnostics"]),
                                  diagnostics_path=os.path.join(WORK, "dev_eval_diagnostics.json"))
            print("COMPETITION METRIC (public eval, %d tasks): %.4f  [cross-check %.4f]" % (
                DEV_RESULT["n_tasks"], DEV_RESULT["score"], competition_score(dev_sub, eval_sol)))
            print("outputs solved: %d / %d | solver errors %d | exact programs found on %d tasks" % (
                DEV_RESULT["n_correct_outputs"], DEV_RESULT["n_test_outputs"], dev_summary["n_errors"],
                dev_summary["n_exact_found"]))
            print("per family:", json.dumps(DEV_RESULT["per_family"]))
            print("near misses:", json.dumps({k: v for k, v in DEV_RESULT["error_analysis"].items()
                                              if k.startswith("near_miss_") and k != "near_miss_task_ids"}))
            if test_challenges:  # reuse the dev attempts for overlapping ids
                write_json_atomic(repair_submission(dev_sub, test_challenges), SUBMISSION)
        except Exception:
            traceback.print_exc()
    elif eval_ch:
        fb = fallback_submission(eval_ch)
        print("no solver available; fallback-only metric: %.4f" % score_submission(fb, eval_sol))

In [ ]:
# ---- final validation of submission.json (repairs malformed entries from the fallbacks)
if test_challenges:
    final = load_json(SUBMISSION) if os.path.isfile(SUBMISSION) else {}
    errs = validate_submission(final, test_challenges)
    if errs:
        print("repairing %d problems, e.g." % len(errs), errs[:5])
        write_json_atomic(repair_submission(final, test_challenges), SUBMISSION)
        errs = validate_submission(load_json(SUBMISSION), test_challenges)
    print("submission.json: %d tasks, valid = %s, %.1f min elapsed" % (
        len(test_challenges), not errs, (time.time() - T0) / 60.0))
    if errs:
        raise RuntimeError("submission.json is invalid: %s" % errs[:5])
else:
    print("no test challenges found; nothing to validate")

In [ ]:
# ---- dev / commit gate: a broken setup fails this run loudly, so the version cannot be submitted.
# Never active in a competition rerun (there the fallback / symbolic submission above stands).
GATE_PROBLEMS = []
if not IS_RERUN:
    if not CODE_OK:
        GATE_PROBLEMS.append("the arcjepa code dataset was not found or does not import (looked in %s)"
                             % [c for c in CODE_CANDIDATES if c])
    elif not RUNNER_OK:
        GATE_PROBLEMS.append("arcjepa.utils.kaggle_submit_runner does not import")
    if RUNNER_OK and not PKG_DIR:
        GATE_PROBLEMS.append("no trained package (arc_jepa_pkg: config.json + model.safetensors) under /kaggle/input;"
                             " is the poby7722/arc-jepa-train output attached and complete?")
    if PKG_DIR and PKG_INFO.get("debug") and not ALLOW_DEBUG_PKG:
        GATE_PROBLEMS.append("the package %s was trained with the debug config (%s parameters): the training kernel"
                             " ran in smoke mode" % (PKG_DIR, PKG_INFO.get("n_parameters")))
    if PKG_DIR and RUNNER_OK:
        frac = DEV_SUMMARY.get("model_loaded_fraction") if DEV_SUMMARY else None
        loaded = None if frac is None else frac > 0
        if loaded is None:  # no dev run / no task finished: try one load here
            try:
                loaded = load_model_package(PKG_DIR, "cpu") is not None
            except Exception:
                traceback.print_exc()
                loaded = False
        if not loaded:
            GATE_PROBLEMS.append("no solver process could load the package %s (see the warnings above)" % PKG_DIR)
    if GATE_PROBLEMS:
        GATE_MSG = "DEV GATE FAILED (this version must not be submitted):\n - " + "\n - ".join(GATE_PROBLEMS)
        if ALLOW_SYMBOLIC:
            print(GATE_MSG + "\n(ARCJEPA_ALLOW_SYMBOLIC=1: reported, not raised)")
        else:
            raise RuntimeError(GATE_MSG)
    else:
        print("dev gate passed: code OK, package %s (%s parameters) loaded" % (PKG_DIR, PKG_INFO.get("n_parameters")))